<a href="https://colab.research.google.com/github/Arthxzs/Analise_Eleitoral_2026/blob/main/An%C3%A1lise_vota%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Montagem do drive para acessar os arquivos

In [1]:
from google.colab import drive
import os

drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/Colab Notebooks/Votacao/data")

Mounted at /content/drive


Montagem dos dois DF

In [8]:
import pandas as pd

usecols1 = ["NR_TURNO", "SG_UF", "SG_UE", "NM_UE", "NM_MUNICIPIO", "QT_APTOS", "QT_TOTAL_SECOES", "QT_COMPARECIMENTO", "QT_VOTOS_BRANCOS", "DS_CARGO", "QT_TOTAL_VOTOS_ANULADOS", "QT_ABSTENCOES", "QT_TOTAL_VOTOS_VALIDOS", "QT_VOTOS_NULOS"] # Selecionando colunas relevantes para a análise
df1 = pd.read_csv("detalhe_votacao_munzona_2026_BRASIL.csv", encoding="ISO-8859-1", sep=";", usecols=usecols1) #Detalhes da votação

usecols2 = ["NM_TIPO_ELEICAO", "SG_UF", "SG_UE", "NM_UE", "NR_TURNO", "DS_ELEICAO", "NM_MUNICIPIO", "DS_CARGO", "QT_VOTOS_NOMINAIS", "NR_CANDIDATO", "NM_CANDIDATO", "NM_PARTIDO", "QT_VOTOS_NOMINAIS_VALIDOS"]
df2 = pd.read_csv("votacao_candidato_munzona_2026_BRASIL.csv", encoding="ISO-8859-1", sep=";", usecols=usecols2) # Números de votação dos candidatos



In [25]:
#numeros_turno_1 = ((df1["DS_CARGO"] == "Governador") & (df1["SG_UF"] == "RJ"))
numeros_t1 = df1.groupby(["SG_UF", "DS_CARGO"])[["QT_TOTAL_VOTOS_VALIDOS", "QT_TOTAL_VOTOS_ANULADOS", "QT_VOTOS_NULOS", "QT_VOTOS_BRANCOS"]].sum(numeric_only=True).sort_values(["QT_TOTAL_VOTOS_ANULADOS"], ascending=[False])


display(numeros_t1.head(5))

,,QT_TOTAL_VOTOS_VALIDOS,QT_TOTAL_VOTOS_ANULADOS,QT_VOTOS_NULOS,QT_VOTOS_BRANCOS
SG_UF,DS_CARGO,,,,
PI,Senador,3640949,22734,506975,299136
SC,Deputado Estadual,4271084,16769,110186,235529
MG,Deputado Federal,11380151,6921,453843,699059
PE,Deputado Estadual,5248293,6679,253416,393817
PI,Deputado Federal,2109063,3634,52380,71640


Números de votação primeiro turno

In [4]:
votacao_cargo_t1 = (df2["NR_TURNO"] == 1) & (df2["DS_CARGO"] == "Presidente") # & (df2["SG_UF"] == "ZZ")

votacao_t1 = df2[votacao_cargo_t1].groupby(["NR_TURNO", "NR_CANDIDATO", "DS_CARGO", "NM_CANDIDATO"])[["QT_VOTOS_NOMINAIS", "QT_VOTOS_NOMINAIS_VALIDOS"]].sum().sort_values("QT_VOTOS_NOMINAIS_VALIDOS", ascending= False)
votacao_t1 = votacao_t1.rename(columns={"QT_VOTOS_NOMINAIS": "Votos totais", "QT_VOTOS_NOMINAIS_VALIDOS": "Votos válidos",  "NR_CANDIDATO": "Número"})
votacao_t1 = votacao_t1.rename_axis(index={"SG_UF": "UF", "NR_TURNO": "Turno","NM_CANDIDATO": "Candidato", "DS_CARGO": "Cargo"})
votacao_t1_colunas = votacao_t1.reset_index()

votacao_turno_1 = votacao_t1_colunas[votacao_t1_colunas["Votos válidos"] > 50000000]
display(votacao_turno_1)

,Turno,NR_CANDIDATO,Cargo,Candidato,Votos totais,Votos válidos
0,1,22,Presidente,FLAVIO NANTES BOLSONARO,56104503,56104503
1,1,13,Presidente,LUIZ INÁCIO LULA DA SILVA,53879538,53879538


In [5]:
votacao_cargo_t2 = (df2["NR_TURNO"] == 2) & (df2["DS_CARGO"] == "Presidente")

votacao_t2 = df2[votacao_cargo_t2].groupby(["NR_TURNO", "DS_CARGO", "NR_CANDIDATO", "NM_CANDIDATO"])[["QT_VOTOS_NOMINAIS_VALIDOS"]].sum().sort_values("QT_VOTOS_NOMINAIS_VALIDOS", ascending=False)
votacao_t2 = votacao_t2.rename(columns={"QT_VOTOS_NOMINAIS_VALIDOS": "Votos válidos"})
votacao_t2 = votacao_t2.rename_axis(index={"NR_TURNO": "Turno", "DS_CARGO": "Cargo", "NR_CANDIDATO": "Número", "NM_CANDIDATO": "Candidato"})

votacao_turno_2 = votacao_t1["Votos válidos"]
display(votacao_t2)

Votos válidos
Turno Cargo      Número Candidato                               
2     Presidente 13     LUIZ INÁCIO LULA DA SILVA       64256232
                 22     FLAVIO NANTES BOLSONARO         59826041

In [6]:
votacao_total_t1 = votacao_turno_1.groupby(["NR_CANDIDATO"])[["Votos válidos"]].sum(numeric_only=True)
display(votacao_total_t1)

,Votos válidos
NR_CANDIDATO,
13,53879538
22,56104503
